# 03 - Gold Layer: Business Aggregates

## What this notebook does
Builds the final, business-consumable gold tables that power the dashboard and
Genie space: pipeline funnel, win rate by lead source, rep performance, deal
velocity, and bookings trend. Each table is pre-aggregated so dashboard queries
are fast and need no joins at query time - the same pattern as Project 1's gold
layer.

## A note on sample size
This project's underlying data is synthetic and modest in volume (400
opportunities). Earlier profiling of the generated dataset showed that win-rate
differences by lead source and by rep are partly sampling noise at this size -
some differences that look meaningful are within normal variation for a few
dozen closed deals per group. The win-rate tables below deliberately include
the underlying closed-deal count alongside every rate, so the dashboard can't
show a rate without also showing how much data backs it.

## Tables created
- `main.sales_revenue_analytics.gold_pipeline_funnel`
- `main.sales_revenue_analytics.gold_win_rate_by_lead_source`
- `main.sales_revenue_analytics.gold_rep_performance`
- `main.sales_revenue_analytics.gold_deal_velocity`
- `main.sales_revenue_analytics.gold_bookings_trend`

In [0]:
%sql
CREATE OR REPLACE TABLE main.sales_revenue_analytics.gold_pipeline_funnel
COMMENT 'Opportunity funnel: count and value of opportunities at each stage, ordered by typical progression through the sales process'
AS
SELECT
  stage,
  CASE stage
    WHEN 'Prospecting' THEN 1
    WHEN 'Qualification' THEN 2
    WHEN 'Needs Analysis' THEN 3
    WHEN 'Value Proposition' THEN 4
    WHEN 'Id. Decision Makers' THEN 5
    WHEN 'Perception Analysis' THEN 6
    WHEN 'Proposal/Price Quote' THEN 7
    WHEN 'Negotiation/Review' THEN 8
    WHEN 'Closed Won' THEN 9
    WHEN 'Closed Lost' THEN 10
    ELSE 99
  END AS stage_order,
  CASE
    WHEN stage = 'Closed Won' THEN 'Closed Won'
    WHEN stage = 'Closed Lost' THEN 'Closed Lost'
    ELSE 'Open'
  END AS stage_type,
  COUNT(*) AS opportunity_count,
  SUM(amount) AS total_amount,
  ROUND(AVG(amount), 2) AS avg_amount,
  SUM(CASE WHEN amount IS NULL THEN 1 ELSE 0 END) AS opportunities_missing_amount
FROM main.sales_revenue_analytics.silver_opportunity
GROUP BY stage
ORDER BY stage_order;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE main.sales_revenue_analytics.gold_win_rate_by_lead_source
COMMENT 'Win rate by lead source among closed deals, with the closed-deal count shown alongside every rate since sample sizes vary widely by source'
AS
SELECT
  COALESCE(lead_source, '(none)') AS lead_source,
  COUNT(*) AS closed_count,
  SUM(CASE WHEN is_won THEN 1 ELSE 0 END) AS won_count,
  ROUND(SUM(CASE WHEN is_won THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS win_rate_pct,
  SUM(CASE WHEN is_won THEN amount ELSE 0 END) AS won_amount
FROM main.sales_revenue_analytics.silver_opportunity
WHERE is_closed = true
GROUP BY COALESCE(lead_source, '(none)')
ORDER BY closed_count DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE main.sales_revenue_analytics.gold_rep_performance
COMMENT 'Rep performance among closed deals: win rate and won-dollar volume, both shown since win rate alone is largely sampling noise at this data volume'
AS
SELECT
  sales_rep,
  COUNT(*) AS closed_count,
  SUM(CASE WHEN is_won THEN 1 ELSE 0 END) AS won_count,
  ROUND(SUM(CASE WHEN is_won THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS win_rate_pct,
  SUM(CASE WHEN is_won THEN amount ELSE 0 END) AS won_amount,
  ROUND(AVG(CASE WHEN is_won THEN amount END), 2) AS avg_won_deal_size
FROM main.sales_revenue_analytics.silver_opportunity
WHERE is_closed = true
GROUP BY sales_rep
ORDER BY won_amount DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE main.sales_revenue_analytics.gold_deal_velocity
COMMENT 'Sales cycle length (days from creation to close) for won deals, overall and by lead source and rep, with counts shown alongside every average'
AS
WITH won_deals AS (
  SELECT
    sales_rep,
    COALESCE(lead_source, '(none)') AS lead_source,
    DATEDIFF(close_date, DATE(created_at)) AS cycle_days
  FROM main.sales_revenue_analytics.silver_opportunity
  WHERE is_won = true
)
SELECT 'Overall' AS dimension, 'All' AS dimension_value,
      COUNT(*) AS won_count,
      ROUND(AVG(cycle_days), 1) AS avg_cycle_days,
      PERCENTILE(cycle_days, 0.5) AS median_cycle_days,
      MIN(cycle_days) AS min_cycle_days,
      MAX(cycle_days) AS max_cycle_days
FROM won_deals

UNION ALL

SELECT 'Sales Rep', sales_rep,
      COUNT(*), ROUND(AVG(cycle_days), 1), PERCENTILE(cycle_days, 0.5),
      MIN(cycle_days), MAX(cycle_days)
FROM won_deals
GROUP BY sales_rep

UNION ALL

SELECT 'Lead Source', lead_source,
      COUNT(*), ROUND(AVG(cycle_days), 1), PERCENTILE(cycle_days, 0.5),
      MIN(cycle_days), MAX(cycle_days)
FROM won_deals
GROUP BY lead_source

ORDER BY dimension, won_count DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE main.sales_revenue_analytics.gold_bookings_trend
COMMENT 'Monthly bookings: count and total value of deals closed as Won, by close month'
AS
SELECT
  DATE_TRUNC('MONTH', close_date) AS booking_month,
  COUNT(*) AS deals_won,
  SUM(amount) AS bookings_amount,
  ROUND(AVG(amount), 2) AS avg_deal_size
FROM main.sales_revenue_analytics.silver_opportunity
WHERE is_won = true
GROUP BY DATE_TRUNC('MONTH', close_date)
ORDER BY booking_month;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM main.sales_revenue_analytics.gold_bookings_trend;

booking_month,deals_won,bookings_amount,avg_deal_size
2025-01-01T00:00:00.000Z,1,246500.0,246500.0
2025-02-01T00:00:00.000Z,2,331500.0,165750.0
2025-03-01T00:00:00.000Z,2,140000.0,70000.0
2025-04-01T00:00:00.000Z,5,195500.0,39100.0
2025-05-01T00:00:00.000Z,4,832500.0,208125.0
2025-06-01T00:00:00.000Z,7,710000.0,101428.57
2025-07-01T00:00:00.000Z,5,258000.0,51600.0
2025-08-01T00:00:00.000Z,5,697500.0,139500.0
2025-09-01T00:00:00.000Z,3,573500.0,191166.67
2025-10-01T00:00:00.000Z,6,1142000.0,190333.33


In [0]:
%sql
SELECT 'gold_pipeline_funnel' AS table_name, COUNT(*) AS row_count FROM main.sales_revenue_analytics.gold_pipeline_funnel
UNION ALL
SELECT 'gold_win_rate_by_lead_source', COUNT(*) FROM main.sales_revenue_analytics.gold_win_rate_by_lead_source
UNION ALL
SELECT 'gold_rep_performance', COUNT(*) FROM main.sales_revenue_analytics.gold_rep_performance
UNION ALL
SELECT 'gold_deal_velocity', COUNT(*) FROM main.sales_revenue_analytics.gold_deal_velocity
UNION ALL
SELECT 'gold_bookings_trend', COUNT(*) FROM main.sales_revenue_analytics.gold_bookings_trend;

table_name,row_count
gold_pipeline_funnel,10
gold_win_rate_by_lead_source,8
gold_rep_performance,6
gold_deal_velocity,15
gold_bookings_trend,21
